# 🎓 Session 1 Lab: Build Your First University Chatbot
### CampusBot: A small rule-based university helpdesk assistant

**Module:** Chatbot System Design  
**Level:** Beginner | **Suggested duration:** 40 to 45 minutes  
**Tools:** Google Colab and Python. No datasets, accounts, APIs, or AI models are required for the core lab.

**Your mission:** Build a chatbot that responds to simple university questions about admissions, the library, classes, Wi-Fi, and student services.

> **Teaching example:** CampusBot represents a **fictional university**. Its answers are illustrative, not official university policies. Replace them with verified information before using it in a real institution.

## 1. Goal and learning outcomes

By the end of this short lab, you will be able to:

1. Explain what a **rule-based chatbot** is.
2. Define a few user **intents** and matching **keywords**.
3. Process user input, choose a response, and handle unknown questions.
4. Test your chatbot and improve its rules.
5. Recognize what a rule-based chatbot **cannot** do.

**Suggested flow:** Warm-up (5 min) → Build (15 min) → Try it (5 min) → Your tasks (15 min) → Discussion (5 min).

## 2. Case study: Help a new student

Imagine a new student arriving at **Sample University**. They may ask:

- “Where can I find my timetable?”
- “How do I apply?”
- “Is there a library?”
- “I can't connect to Wi-Fi.”
- “How can I contact Student Services?”

Instead of answering every possible question, your first chatbot will recognise a few **intents**, meaning the goals behind a message.

| Example message | Intent | Example response |
|:--|:--|:--|
| “Hello” | `greeting` | Welcome the student |
| “How can I apply?” | `admissions` | Direct them to admissions |
| “Where is my class schedule?” | `timetable` | Direct them to the student portal |
| “What about the library?” | `library` | Direct them to library information |
| “Wi-Fi does not work” | `it_support` | Refer them to the IT helpdesk |

These sample answers intentionally avoid making up campus addresses, deadlines, or opening hours.

## 3. How a rule-based chatbot works

```text
Student message
      ↓
Clean the message (lowercase, remove punctuation)
      ↓
Match words or short phrases against rules
      ↓
Recognised intent? ── No ──→ Safe fallback response
      │
     Yes
      ↓
Choose the prepared answer
      ↓
Display the answer
```

**Important:** This bot does **not** train on data or use a large language model. It follows rules that **you** write.

## 4. Warm-up: Create a two-rule chatbot

**The purpose of this code:** See how a few `if` and `elif` statements can produce a basic chatbot response.

Run the next cell. Observe what happens for a message that was not anticipated.

In [ ]:
def tiny_chatbot(message):
    message = message.lower()

    if "hello" in message:
        return "Hello! Welcome to Sample University."
    elif "library" in message:
        return "Please check the university library webpage."
    else:
        return "Sorry, I do not know the answer yet."

print("You: Hello")
print("CampusBot:", tiny_chatbot("Hello"))
print("You: Where is the library?")
print("CampusBot:", tiny_chatbot("Where is the library?"))
print("You: Can you help with Wi-Fi?")
print("CampusBot:", tiny_chatbot("Can you help with Wi-Fi?"))

You: Hello
CampusBot: Hello! Welcome to Sample University.
You: Where is the library?
CampusBot: Please check the university library webpage.
You: Can you help with Wi-Fi?
CampusBot: Sorry, I do not know the answer yet.


**Think for 30 seconds:** Why did CampusBot fail to answer the Wi-Fi question? What would you add so it could answer more topics?

We now move the rules into a dictionary, making the chatbot easier to extend.

## 5. Step 1: Clean student messages

Different users may type `HELLO`, `hello!`, or `Hello`. We want them to be treated similarly.

**The purpose of this code:** Create a function that converts text to lowercase and removes punctuation.

In [ ]:
import re  # Built-in Python library for working with text patterns

def normalize(message):
    """Return a simple, clean version of the student's message."""
    message = message.lower()
    message = re.sub(r"[^a-z0-9\s]", " ", message)  # Punctuation → spaces
    message = re.sub(r"\s+", " ", message)         # Extra spaces → one space
    return message.strip()

for example in ["HELLO!", "  Library???  ", "Wi-Fi is NOT working!"]:
    print(f"Original: {example!r} → Clean: {normalize(example)!r}")

Original: 'HELLO!' → Clean: 'hello'
Original: '  Library???  ' → Clean: 'library'
Original: 'Wi-Fi is NOT working!' → Clean: 'wi fi is not working'


**Quick check:** Notice that `Wi-Fi` becomes `wi fi`. The rules below include both `wifi` and `wi fi` to handle these variations.

## 6. Step 2: Create the university chatbot's rules

Every rule has three elements:

- An **intent name**, for example `library`.
- A few **keywords or phrases** that suggest that intent.
- A **prepared response** returned to the student.

**The purpose of this code:** Store the chatbot's knowledge in one small Python dictionary. You can edit this dictionary later without rewriting the chatbot logic.

In [ ]:
# Rule order matters: specific help requests come before general greetings.
RULES = {
    "admissions": {
        "keywords": ["admission", "apply", "application", "enroll", "enrol"],
        "response": "For application information, please contact the admissions team or visit the university admissions webpage."
    },
    "timetable": {
        "keywords": ["timetable", "schedule", "class time", "my class", "lecture time"],
        "response": "You can find your course timetable in the student portal."
    },
    "library": {
        "keywords": ["library", "borrow book", "books"],
        "response": "For opening hours and book loans, please check the university library webpage."
    },
    "it_support": {
        "keywords": ["wifi", "wi fi", "internet", "password", "login", "log in"],
        "response": "For internet or account issues, please contact the university IT helpdesk."
    },
    "student_services": {
        "keywords": ["student services", "student office", "contact"],
        "response": "You can reach Student Services through the student portal or the university website."
    },
    "goodbye": {
        "keywords": ["bye", "goodbye", "quit", "exit"],
        "response": "Goodbye! Have a great day."
    },
    "greeting": {
        "keywords": ["hello", "hi", "hey", "good morning"],
        "response": "Hello! I am CampusBot. Ask me about admissions, timetables, the library, Wi-Fi, or Student Services."
    }
}

print("CampusBot has", len(RULES), "rules.")
print("Available intents:", ", ".join(RULES))

CampusBot has 7 rules.
Available intents: admissions, timetable, library, it_support, student_services, goodbye, greeting


## 7. Step 3: Recognise the student's intent

**The purpose of this code:** Compare the cleaned message with your keywords and return the first matching intent.

We put spaces around both the message and each keyword. This helps match complete words: `hi` should match `Hi!`, but **not** the `hi` inside `this`.

This is still a deliberately simple matching strategy. It is not semantic understanding, and **the first matching rule wins**.

In [ ]:
def detect_intent(message):
    """Find the first matching intent, or return None if none matches."""
    clean_message = normalize(message)
    padded_message = f" {clean_message} "

    for intent, rule in RULES.items():
        for keyword in rule["keywords"]:
            padded_keyword = f" {normalize(keyword)} "
            if padded_keyword.strip() and padded_keyword in padded_message:
                return intent

    return None

examples = ["Hello!", "Where is the library?", "Can I apply?", "I need WI-FI!", "This is new.","How do I connect the WiFi","need to borrow some BOOKS!!!"]
for message in examples:
    print(f"{message!r:28} -> {detect_intent(message)}")

'Hello!'                     -> greeting
'Where is the library?'      -> library
'Can I apply?'               -> admissions
'I need WI-FI!'              -> it_support
'This is new.'               -> None
'How do I connect the WiFi'  -> it_support
'need to borrow some BOOKS!!!' -> library


## 8. Step 4: Generate a reply (including a fallback)

A real chatbot must also respond when it does not recognise the message. Do not invent an answer when no rule matches.

**The purpose of this code:** Convert an intent into a prepared response, or display a polite fallback.

In [ ]:
FALLBACK = (
    "Sorry, I cannot answer that yet. "
    "Try asking about admissions, timetables, the library, Wi-Fi, or Student Services."
)

def chatbot_reply(message):
    if not normalize(message):
        return "Please type a university question so I can try to help."

    intent = detect_intent(message)
    if intent is None:
        return FALLBACK

    return RULES[intent]["response"]

print("CampusBot:", chatbot_reply("How can I see my timetable?"))
print("CampusBot:", chatbot_reply("Can you predict my exam grade?"))

CampusBot: You can find your course timetable in the student portal.
CampusBot: Sorry, I cannot answer that yet. Try asking about admissions, timetables, the library, Wi-Fi, or Student Services.


## 9. Step 5: Test CampusBot with different messages

**The purpose of this code:** Try several messages without typing them one by one. Notice how the same intent can be triggered by different words.

In [ ]:
test_messages = [
    "Good morning!",
    "How can I APPLY?",
    "Where is my schedule?",
    "Is the library open?",
    "My Wi-Fi is not working!",
    "How do I contact Student Services?",
    "Do you know the cafeteria menu?",
    "Bye!",
    "need to buy some books",
    "is the cafeteria open yet?",
    "when is my timetable going to be upload?"
]

for message in test_messages:
    print("You:", message)
    print("CampusBot:", chatbot_reply(message))
    print()

You: Good morning!
CampusBot: Hello! I am CampusBot. Ask me about admissions, timetables, the library, Wi-Fi, or Student Services.

You: How can I APPLY?
CampusBot: For application information, please contact the admissions team or visit the university admissions webpage.

You: Where is my schedule?
CampusBot: You can find your course timetable in the student portal.

You: Is the library open?
CampusBot: For opening hours and book loans, please check the university library webpage.

You: My Wi-Fi is not working!
CampusBot: For internet or account issues, please contact the university IT helpdesk.

You: How do I contact Student Services?
CampusBot: You can reach Student Services through the student portal or the university website.

You: Do you know the cafeteria menu?
CampusBot: Sorry, I cannot answer that yet. Try asking about admissions, timetables, the library, Wi-Fi, or Student Services.

You: Bye!
CampusBot: Goodbye! Have a great day.

You: need to buy some books
CampusBot: For op

### Quick quality check

**The purpose of this code:** Confirm that expected basic behaviour works, including an unknown message and a complete-word match.

In [ ]:
assert detect_intent("Where is the library?") == "library"
assert detect_intent("My WI-FI isn't working") == "it_support"
assert detect_intent("This is an unknown topic") is None  # 'hi' must not match inside 'this'
assert chatbot_reply("  ").startswith("Please type")
print("✅ The four basic checks passed.")

✅ The four basic checks passed.


## 10. Step 6: Chat with your bot

**The purpose of this code:** Start a simple conversation in Colab using `input()` and `print()`.

Run the next two cells. Type a few messages, then enter `bye`, `goodbye`, `exit`, or `quit` to finish.

> If you accidentally keep the chat running, use Colab's stop button to interrupt the cell.

In [ ]:
def start_chat():
    print("CampusBot: Hello! Ask me a university question. Type 'bye' to finish.")
    while True:
        message = input("You: ")
        answer = chatbot_reply(message)
        print("CampusBot:", answer)

        if detect_intent(message) == "goodbye":
            break

In [ ]:
# Run this cell to have a real conversation.
start_chat()

CampusBot: Hello! Ask me a university question. Type 'bye' to finish.
You: library? or schedule?
CampusBot: You can find your course timetable in the student portal.
You: and the library?
CampusBot: For opening hours and book loans, please check the university library webpage.
You: I need to apply
CampusBot: For application information, please contact the admissions team or visit the university admissions webpage.
You: the staff at uni?
CampusBot: Sorry, I cannot answer that yet. Try asking about admissions, timetables, the library, Wi-Fi, or Student Services.
You: what about the staff?
CampusBot: Sorry, I cannot answer that yet. Try asking about admissions, timetables, the library, Wi-Fi, or Student Services.
You: what about the student services?
CampusBot: You can reach Student Services through the student portal or the university website.
You: ok, bye!
CampusBot: Goodbye! Have a great day.


## 11. Your turn: Complete three mini tasks (15 minutes)

**Work alone or in pairs.** Keep using the same CampusBot. You do not need to create a new chatbot.

### How to complete the exercises

1. Find the code cell directly below each task heading.
2. Look for `# TODO` comments and the placeholder text. **Those are the missing parts you must write.**
3. Edit the code, run that cell, and observe the chatbot's response.
4. After completing all three tasks, run **Check your answers** at the end of this section.

| Task | What you will edit | Time |
|:--|:--|:--|
| **1. Improve an answer** | One response string | 3 min |
| **2. Add a new intent** | Keywords and a response | 6 min |
| **3. Recognize new expressions** | Add two phrases to an existing list | 6 min |

**Important:** The student version starts with incomplete task code intentionally. It can run before you solve the exercises, but its test messages will not all succeed until you complete the `TODO` parts.

### Task 1: Improve the library answer

CampusBot currently gives a generic answer about the library. Change its response to mention **both book borrowing and study spaces**. Do not invent opening hours, locations, or university policies.

**The purpose of this code:** Modify a response stored in a Python dictionary and see the difference immediately.

**Your missing code:** Replace the text `TODO: Write your improved response here.` with a full sentence that includes the words **borrowing** and **study** (or close variants).


In [ ]:
# TASK 1: Complete the missing response text.
# The sentence should mention BOTH book borrowing and study spaces.

print("Before:", chatbot_reply("I have a library question"))

# TODO 1: Replace the entire text between the quotation marks.
RULES["library"]["response"] = "If you would you like to borrow a book or check the study spaces, please check the university library webpage."

print("After: ", chatbot_reply("I have a library question"))


Before: If you would you like to borrow a book or check the opening hours, please check the university library webpage.
After:  If you would you like to borrow a book or check the study spaces, please check the university library webpage.


### Task 2: Add a cafeteria intent

New students may ask about places to eat on campus. Extend the existing `RULES` dictionary with a new intent named `cafeteria`.

Your chatbot should recognize **both** of these student messages:

- `Where is the cafeteria?`
- `Is the canteen open?`

**The purpose of this code:** Define a new intent using a list of keywords and a fixed response.

**Your missing code:** In the next cell, complete the two `TODO` fields:

1. Replace the empty `keywords` list with the two expressions `"cafeteria"` and `"canteen"`.
2. Replace the placeholder `response` with a helpful sentence that refers the student to official dining information. Do not invent hours or locations.

**Hint:** A Python list of two strings looks like `["first", "second"]`.


In [ ]:
# TASK 2: Create the new cafeteria intent.
RULES["cafeteria"] = {
    "keywords": ["cafeteria","canteen"],  # TODO 2A: Add "cafeteria" and "canteen".
    "response": "To check the cafeteria menu, the opening hours and the location, please check the univeristy cafeteria website."  # TODO 2B: Replace this text.
}

print("Test A:", chatbot_reply("Where is the cafeteria?"))
print("Test B:", chatbot_reply("Is the canteen open?"))


Test A: To check the cafeteria menu, the opening hours and the location, please check the univeristy cafeteria website.
Test B: To check the cafeteria menu, the opening hours and the location, please check the univeristy cafeteria website.


### Task 3: Recognize more IT support questions

CampusBot already recognizes questions mentioning `Wi-Fi`, `internet`, or `password`. However, a student may ask using other words.

Your chatbot should also recognize:

- `I need tech support`
- `I have a computer problem`

**The purpose of this code:** Improve a rule-based chatbot by adding new phrases to an existing intent.

**Your missing code:** Replace the empty list `[]` with these two exact strings: `"tech support"` and `"computer problem"`.

**Hint:** You do not need to change `detect_intent()` or `chatbot_reply()`. The existing `.extend()` statement adds the new phrases to the `it_support` rule.


In [ ]:
# TASK 3: Add two ways to ask for IT help.
# TODO 3: Replace the empty list with two strings.
new_keywords = ["tech support","computer problem"]

# Update the existing rule. This loop avoids duplicates if you rerun the cell.
for keyword in new_keywords:
    if keyword not in RULES["it_support"]["keywords"]:
        RULES["it_support"]["keywords"].append(keyword)

print("Test A:", chatbot_reply("I need tech support"))
print("Test B:", chatbot_reply("I have a computer problem"))


Test A: For internet or account issues, please contact the university IT helpdesk.
Test B: For internet or account issues, please contact the university IT helpdesk.


### Check your answers

**The purpose of this code:** Verify each exercise after you have completed it.

Run this code cell after Tasks 1, 2, and 3. If a check says `INCOMPLETE`, return to that task, edit the `TODO` code, rerun that task cell, and run the checks again.

The tests check a few key requirements. You should also read each answer to make sure it is clear and does not claim unverified opening hours or locations.


In [ ]:
# SELF-CHECK: No editing required in this cell.
library_answer = RULES["library"]["response"].lower()
cafeteria_rule = RULES.get("cafeteria", {})
cafeteria_answer = cafeteria_rule.get("response", "")

checks = {
    "Task 1 | Response is no longer a placeholder": "todo:" not in library_answer,
    "Task 1 | Mentions borrowing and study spaces": (
        "borrow" in library_answer and "study" in library_answer
    ),
    "Task 2 | Keywords are supplied": {"cafeteria", "canteen"}.issubset(
        set(cafeteria_rule.get("keywords", []))
    ),
    "Task 2 | Recognizes cafeteria": detect_intent("Where is the cafeteria?") == "cafeteria",
    "Task 2 | Recognizes canteen": detect_intent("Is the canteen open?") == "cafeteria",
    "Task 2 | Has a completed response": bool(cafeteria_answer.strip()) and "todo:" not in cafeteria_answer.lower(),
    "Task 3 | Recognizes tech support": detect_intent("I need tech support") == "it_support",
    "Task 3 | Recognizes computer problem": detect_intent("I have a computer problem") == "it_support",
}

for description, passed in checks.items():
    print(("PASS" if passed else "INCOMPLETE") + " | " + description)
print(f"\nChecks completed: {sum(checks.values())}/{len(checks)}")


PASS | Task 1 | Response is no longer a placeholder
PASS | Task 1 | Mentions borrowing and study spaces
PASS | Task 2 | Keywords are supplied
PASS | Task 2 | Recognizes cafeteria
PASS | Task 2 | Recognizes canteen
PASS | Task 2 | Has a completed response
PASS | Task 3 | Recognizes tech support
PASS | Task 3 | Recognizes computer problem

Checks completed: 8/8


## 12. Discuss: What are CampusBot's limitations?

Try the message **“Hi! I have two questions about the library and my timetable.”**

Discuss briefly:

1. Which intent does CampusBot choose? Why?
2. Can it understand completely new ways of phrasing a request?
3. What happens if your sample university changes its procedures?
4. Why would a real university chatbot need verified answers, privacy protection, and a route to human support?

**Key idea:** This is a **rule-based system**. You manually define the intents and answers. It can be predictable and easy to test, but its coverage is limited by the rules you write.

## 13. Optional extension: Give CampusBot a chat interface

If you finish early, you can replace the text-based console with a small web interface using **Gradio**.

Run the next two cells only if you want this extension. The core chatbot works **without** Gradio.

**The purpose of this code:** Reuse your existing chatbot logic inside a visual chat application. No model or API is involved.

**Privacy reminder:** Only enter made-up or non-sensitive example questions into a classroom demo.

In [ ]:
%pip -q install gradio

In [ ]:
import gradio as gr

def reply_in_interface(message, history):
    # Gradio passes the chat history, but our simple bot does not use it.
    return chatbot_reply(message)

demo = gr.ChatInterface(
    fn=reply_in_interface,
    title="🎓 CampusBot | Sample University",
    description="A small rule-based university chatbot. Try: library, timetable, Wi-Fi, admissions."
)
demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://e6ac7bb64d90086a6f.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## 14. Takeaways and next session

You have built a small chatbot with:

- **Input handling:** Receive a student message.
- **Preprocessing:** Clean the text.
- **Intent recognition:** Match keywords to predefined intents.
- **Response generation:** Return a prepared answer.
- **Fallback:** Handle an unknown request politely.
- **Testing:** Check correct and incorrect cases.

**How is this related to chatbot platforms?** Later, when you explore **Rasa** or **Dialogflow CX**, you will see the same high-level ideas, but with more sophisticated methods for recognising intents, managing conversation state, integrating services, and evaluating interactions.

**Exit question:** What is one benefit and one limitation of a rule-based chatbot?

---
*Student lab. The example university and responses are fictional. The notebook is intended as an introduction, not a production student-support service.*